# L5 M365 Determinism Projection

Projection notebook for the deterministic supported-surface check. Source determinism notebook: `notebooks/m365/INV-M365-C-001-determinism.ipynb`.

In [ ]:
import re
from pathlib import Path

import yaml

registry = yaml.safe_load(Path("registry/capability_registry.yaml").read_text())
implemented = sorted(a["action"] for a in registry["actions"] if a.get("status") == "implemented")
router_text = Path("src/provisioning_api/routers/m365.py").read_text()
match = re.search(r"_SUPPORTED_ACTIONS\\s*=\\s*\\{([^}]*)\\}", router_text, re.S)
router = sorted(re.findall(r"\"([a-z_]+)\"", match.group(1)))
contract = []
for line in Path("docs/CAIO_M365_CONTRACT.md").read_text().splitlines():
    if line.startswith("| `"):
        parts = [p.strip() for p in line.split("|")]
        if len(parts) > 2:
            contract.append(parts[1].strip("`"))
contract = sorted(contract)
surface_a = sorted(set(implemented) & set(router) & set(contract))
surface_b = sorted(set(implemented) & set(router) & set(contract))
assert surface_a == surface_b
assert len(surface_a) == 9
print(surface_a)